# Layer Functional Profiling

Characterize what each layer does: prediction impact, computation type,
information gain, redundancy, and functional summary.

## Why This Matters

Layer functional characterizes what each layer contributes to the model's computation. Understanding layer-level organization — which layers are critical, which are redundant, and how they specialize — is essential for both interpretability and efficient model design.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.layer_functional_profiling import (
    layer_prediction_impact, layer_computation_type,
    layer_information_gain, layer_redundancy_analysis,
    layer_functional_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Prediction Impact

How much does each layer change the model's prediction?

In [ ]:
result = layer_prediction_impact(model, tokens)
for p in result['per_layer']:
    changed = '→' if p['prediction_changed'] else '='
    print(f"  Layer {p['layer']}: tok {p['top_token_before']:3d} {changed} "
          f"{p['top_token_after']:3d}, KL={p['kl_divergence']:.4f}")

## Computation Type

Is each layer attention-dominated, MLP-dominated, or balanced?

In [ ]:
result = layer_computation_type(model, tokens)
for p in result['per_layer']:
    attn_bar = '▓' * int(p['attn_fraction'] * 20)
    mlp_bar = '░' * (20 - int(p['attn_fraction'] * 20))
    print(f"  Layer {p['layer']}: [{p['computation_type']:22s}] "
          f"attn={p['attn_norm']:.4f} mlp={p['mlp_norm']:.4f} {attn_bar}{mlp_bar}")

## Information Gain

How much new information (orthogonal to previous direction) does each layer add?

In [ ]:
result = layer_information_gain(model, tokens)
for p in result['per_layer']:
    bar = '█' * int(p['new_info_fraction'] * 20)
    print(f"  Layer {p['layer']}: parallel={p['parallel_component']:.4f}, "
          f"perp={p['perpendicular_component']:.4f}, "
          f"new_info={p['new_info_fraction']:.1%} {bar}")

## Redundancy

Which layer pairs produce similar outputs?

In [ ]:
result = layer_redundancy_analysis(model, tokens)
print(f"Redundant pairs: {result['n_redundant_pairs']}\n")
for p in result['pairs']:
    flag = ' [REDUNDANT]' if p['is_redundant'] else ''
    print(f"  Layer {p['layer_a']} vs {p['layer_b']}: "
          f"similarity={p['similarity']:.4f}{flag}")

## Functional Summary

One-line summary of each layer's function.

In [ ]:
result = layer_functional_summary(model, tokens)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: attn={p['attn_magnitude']:.4f}, "
          f"mlp={p['mlp_magnitude']:.4f}, "
          f"entropy={p['mean_attn_entropy']:.4f}, "
          f"logit_impact={p['logit_impact']:.4f}")